# Lease Collection Scorecard

End-to-end pipeline for a collection (early-delinquency) scorecard on lease loans:
cleaning → behavioural feature engineering → PSI / missingness / outlier filtering →
WOE binning (`scorecardpy`) → logistic regression → scorecard, score bands and out-of-time validation.

**Data is not included.** Point `DATA_DIR` / `load_table()` at your own data source.
Target: `bad = 1` if the loan reaches 31+ days past due in the outcome window.


Import libraries

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
import seaborn as sns
import matplotlib.ticker as mtick
import matplotlib.pyplot as plt
import warnings
import scorecardpy as sc
pd.set_option('display.max_columns',500)
pd.set_option('display.max_rows',500)
pd.options.display.float_format = '{:,.2f}'.format
warnings.filterwarnings('ignore')
sns.set(style = 'white')
from pathlib import Path

# ── Paths (override for your environment) ──────────────────────────────
DATA_DIR = Path("data")          # input datasets
OUT_DIR = Path("outputs/collection_score")  # model artifacts & reports
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def load_table(name: str) -> pd.DataFrame:
    """Load a dataset by name. Replace with your own data source (DB, warehouse, files)."""
    return pd.read_parquet(DATA_DIR / f"{name}.parquet")

In [ ]:
import os

os.chdir("..")

### Import dataset

In [ ]:
# Expects a loan-level snapshot table with one row per (base_date, loanid),
# behavioural features (base_*, app_*, cdr_*, mobile_*, uloan_*, card_*) and
# target columns `target_days` / `target`.
temp = load_table("collection_base")

In [ ]:
df = temp.copy()

### Basic Cleaning

In [ ]:
df.dropna(subset=["base_date"], inplace=True)
df.dropna(subset=["loanid"], inplace=True)
df.dropna(subset=["target_date"], inplace=True)
df.dropna(subset=["dpd"], inplace=True)
df.drop_duplicates(subset=["base_date", "loanid"], inplace=True)

In [ ]:


df["bad"] = np.where(df["target_days"] >= 31, 1, 0)
df.drop(columns=["target_days", "target"], inplace=True)


In [ ]:
df.head()

Preprocessing-Featurization

In [ ]:
df_target = df.copy()
df_target = df_target.drop_duplicates()

In [ ]:
df_target.shape

In [ ]:
date_cols = [
    c
    for c in df_target.columns
    if c != "base_date" and str(df_target[c].dtype).startswith("datetime")
]
df_target.drop(columns=date_cols, inplace=True)

In [ ]:
df_target.groupby("bad").count()

In [ ]:
# df_target.drop(
#     columns=["userid", "borrowerid", "loanid", "dpd"],
#     inplace=True,
# )

In [ ]:
for i in df_target.columns.tolist():
    try:
        df_target[i] = df_target[i].astype("float")

    except:
        continue

In [ ]:
for c in df_target.select_dtypes(include=np.number).columns:
    if not any(
        x in c.lower()
        for x in [
            "change",
            "rate",
            "momentum",
            "ratio",
            "volatility",
            "acceleration",
            "_std",
            "trend",
            "growth",
            "decay",
            "shift",
            "date",
        ]
    ):
        df_target[c] = df_target[c].clip(lower=0)

In [ ]:
num_cols = df_target.select_dtypes(include="number").columns.tolist()

for c in num_cols:

    # convert type
    df_target[c] = df_target[c].astype(float)

    # 1. recency / since_
    if "recency" in c or "since_" in c:
        max_value = df_target[c].max()
        df_target[c] = df_target[c].fillna(max_value + 1)

    # 2. profile / tenure
    elif "profile" in c or "tenure" in c:
        mean_value = df_target[c].mean()
        df_target[c] = df_target[c].fillna(mean_value)

    # 3. drop unwanted columns
    elif "due" in c or "cdr_phone" in c:
        df_target.drop(columns=c, inplace=True)

    # 4. default
    else:
        df_target[c] = df_target[c].fillna(0)

In [ ]:
df_target.head()

### Build features

In [ ]:
df_target["monthly_principal"] = (
    df_target["loan_principal_amt"] / df_target["loan_duration_months"]
)

In [ ]:
df_target.loc[:, df_target.columns.str.contains("base_")].columns.to_list()

In [ ]:
df_target.head()

In [ ]:
safe_div = lambda a, b: np.where(b == 0, 0, a / b)

df_target["limit_usage"] = (
    df_target["loan_principal_amt"] / df_target["base_first_limit"]
)
df_target["monthly_principal"] = (
    df_target["loan_principal_amt"] / df_target["loan_duration_months"]
)

df_target["base_chronic_late_flag"] = (
    (df_target["base_late_rate_372_188"] > 0.5)
    & (df_target["base_late_rate_187_98"] > 0.5)
    & (df_target["base_late_rate_97_38"] > 0.5)
    & (df_target["base_late_rate_37_8"] > 0.5)
).astype(int)

df_target["base_improving_flag"] = (
    (df_target["base_late_rate_37_8"] < df_target["base_late_rate_97_38"])
).astype(int)

recent_paid = df_target["base_paid_cnt_37_8"] + df_target["base_paid_cnt_97_38"]

historical_paid = (
    df_target["base_paid_cnt_372_188"]
    + df_target["base_paid_cnt_187_98"]
    + df_target["base_paid_cnt_97_38"]
    + df_target["base_paid_cnt_37_8"]
)

df_target["base_recent_payment_concentration"] = safe_div(recent_paid, historical_paid)

df_target["base_late_paid_1_90"] = (
    df_target["base_late_paid_1_7"]
    + df_target["base_late_paid_8_14"]
    + df_target["base_late_paid_15_30"]
    + df_target["base_late_paid_31_60"]
    + df_target["base_late_paid_61_90"]
)
df_target.drop(columns=["base_late_paid_1_7", "base_late_paid_8_14", "base_late_paid_15_30", "base_late_paid_31_60", "base_late_paid_61_90"], inplace=True)

safe_div = lambda a, b: np.where(b == 0, 0, a / b)
safe_diff = lambda a, b: a - b  # signed: positive = worsening (more days late)

# ── DPD trajectory features ───────────────────────────────────────────

# Max DPD trend: recent vs older (positive = getting worse)
df_target["base_max_dpd_chg_37_8_vs_97_38"]    = safe_diff(df_target["base_max_dpd_37_8"],    df_target["base_max_dpd_97_38"])
df_target["base_max_dpd_chg_97_38_vs_187_98"]  = safe_diff(df_target["base_max_dpd_97_38"],   df_target["base_max_dpd_187_98"])
df_target["base_max_dpd_chg_37_8_vs_187_98"]   = safe_diff(df_target["base_max_dpd_37_8"],    df_target["base_max_dpd_187_98"])
df_target["base_max_dpd_chg_37_8_vs_372_188"]  = safe_diff(df_target["base_max_dpd_37_8"],    df_target["base_max_dpd_372_188"])

# Avg DPD trend
df_target["base_avg_dpd_chg_37_8_vs_97_38"]    = safe_diff(df_target["base_avg_dpd_37_8"],    df_target["base_avg_dpd_97_38"])
df_target["base_avg_dpd_chg_97_38_vs_187_98"]  = safe_diff(df_target["base_avg_dpd_97_38"],   df_target["base_avg_dpd_187_98"])
df_target["base_avg_dpd_chg_37_8_vs_187_98"]   = safe_diff(df_target["base_avg_dpd_37_8"],    df_target["base_avg_dpd_187_98"])

# DPD acceleration: is the worsening itself speeding up?
# positive = DPD growing faster recently (2nd derivative)
df_target["base_max_dpd_acceleration"] = (
    df_target["base_max_dpd_chg_37_8_vs_97_38"]
    - df_target["base_max_dpd_chg_97_38_vs_187_98"]
)
df_target["base_avg_dpd_acceleration"] = (
    df_target["base_avg_dpd_chg_37_8_vs_97_38"]
    - df_target["base_avg_dpd_chg_97_38_vs_187_98"]
)

# Max DPD ratio recent/old — how much worse is recent vs historical?
df_target["base_max_dpd_ratio_recent_vs_old"] = safe_div(
    df_target["base_max_dpd_37_8"] + df_target["base_max_dpd_97_38"],
    df_target["base_max_dpd_187_98"] + df_target["base_max_dpd_372_188"] + 1
)

# ── Unpaid trend features ─────────────────────────────────────────────

# Unpaid count change (positive = more unpaid recently = worsening)
df_target["base_unpaid_chg_37_8_vs_97_38"]   = safe_diff(df_target["base_unpaid_cnt_37_8"],   df_target["base_unpaid_cnt_97_38"])
df_target["base_unpaid_chg_97_38_vs_187_98"] = safe_diff(df_target["base_unpaid_cnt_97_38"],  df_target["base_unpaid_cnt_187_98"])
df_target["base_unpaid_chg_37_8_vs_187_98"]  = safe_diff(df_target["base_unpaid_cnt_37_8"],   df_target["base_unpaid_cnt_187_98"])

# Unpaid rate per window (unpaid / total due in that window)
df_target["base_unpaid_rate_372_188"] = safe_div(df_target["base_unpaid_cnt_372_188"], df_target["base_paid_cnt_372_188"] + df_target["base_unpaid_cnt_372_188"])
df_target["base_unpaid_rate_187_98"]  = safe_div(df_target["base_unpaid_cnt_187_98"],  df_target["base_paid_cnt_187_98"]  + df_target["base_unpaid_cnt_187_98"])
df_target["base_unpaid_rate_97_38"]   = safe_div(df_target["base_unpaid_cnt_97_38"],   df_target["base_paid_cnt_97_38"]   + df_target["base_unpaid_cnt_97_38"])
df_target["base_unpaid_rate_37_8"]    = safe_div(df_target["base_unpaid_cnt_37_8"],    df_target["base_unpaid_cnt_37_8"]  + df_target["base_paid_cnt_37_8"])

# Unpaid rate change (positive = more unpaid rate recently)
df_target["base_unpaid_rate_chg_37_8_vs_97_38"]   = safe_diff(df_target["base_unpaid_rate_37_8"],  df_target["base_unpaid_rate_97_38"])
df_target["base_unpaid_rate_chg_97_38_vs_187_98"] = safe_diff(df_target["base_unpaid_rate_97_38"], df_target["base_unpaid_rate_187_98"])
df_target["base_unpaid_rate_chg_187_98_vs_372_188"] = safe_diff(df_target["base_unpaid_rate_187_98"], df_target["base_unpaid_rate_372_188"])


# ── Late severity bucket ratios ───────────────────────────────────────
# What fraction of late payments fall in each severity bucket?
total_late = (
    df_target["base_late_paid_1_90"]
    + df_target["base_late_paid_91_180"]
    + df_target["base_late_paid_181_360"]
    + df_target["base_late_paid_361"]
)

df_target["base_late_severity_mild_ratio"]    = safe_div(df_target["base_late_paid_1_90"],    total_late)  # 1–90d
df_target["base_late_severity_severe_ratio"]  = safe_div(df_target["base_late_paid_91_180"],  total_late)  # 91–180d
df_target["base_late_severity_extreme_ratio"] = safe_div(
    df_target["base_late_paid_181_360"] + df_target["base_late_paid_361"], total_late
)  # 181d+

# ── Repayment momentum ────────────────────────────────────────────────
# Rep count change (positive = paying more frequently recently)
df_target["base_rep_cnt_chg_37_8_vs_97_38"]   = safe_diff(df_target["base_rep_cnt_37_8"],  df_target["base_rep_cnt_97_38"])
df_target["base_rep_cnt_chg_97_38_vs_187_98"] = safe_diff(df_target["base_rep_cnt_97_38"], df_target["base_rep_cnt_187_98"])

# Repayments per invoice (how many payment events per invoice due)
df_target["base_rep_per_invoice_37_8"]  = safe_div(df_target["base_rep_cnt_37_8"],  df_target["base_paid_cnt_37_8"]  + df_target["base_unpaid_cnt_37_8"])
df_target["base_rep_per_invoice_97_38"] = safe_div(df_target["base_rep_cnt_97_38"], df_target["base_paid_cnt_97_38"] + df_target["base_unpaid_cnt_97_38"])

# Change in reps-per-invoice (rising = more partial payment events per invoice)
df_target["base_rep_per_invoice_chg"] = safe_diff(
    df_target["base_rep_per_invoice_37_8"],
    df_target["base_rep_per_invoice_97_38"]
)

# ── Bulk payment trajectory ───────────────────────────────────────────
# Bulk ratio per window
df_target["base_bulk_ratio_37_8"]    = safe_div(df_target["base_bulk_cnt_37_8"],    df_target["base_rep_cnt_37_8"])
df_target["base_bulk_ratio_97_38"]   = safe_div(df_target["base_bulk_cnt_97_38"],   df_target["base_rep_cnt_97_38"])
df_target["base_bulk_ratio_187_98"]  = safe_div(df_target["base_bulk_cnt_187_98"],  df_target["base_rep_cnt_187_98"])
df_target["base_bulk_ratio_372_188"] = safe_div(df_target["base_bulk_cnt_372_188"], df_target["base_rep_cnt_372_188"])

# Bulk ratio change (negative = stopped doing bulk catch-up payments)
df_target["base_bulk_ratio_chg_37_8_vs_97_38"]   = safe_diff(df_target["base_bulk_ratio_37_8"],   df_target["base_bulk_ratio_97_38"])
df_target["base_bulk_ratio_chg_97_38_vs_187_98"] = safe_diff(df_target["base_bulk_ratio_97_38"],  df_target["base_bulk_ratio_187_98"])

# ── Composite risk signals ────────────────────────────────────────────

# Deterioration score: combines DPD increase + pay_freq drop + unpaid increase
# Higher = more signs of deterioration across all dimensions
df_target["base_deterioration_score"] = (
    (df_target["base_max_dpd_chg_37_8_vs_187_98"] > 0).astype(int)      # DPD worsening
    + (df_target["base_pay_freq_chg_37_8_vs_97_38"] < 0).astype(int)    # paying less often
    + (df_target["base_unpaid_chg_37_8_vs_187_98"] > 0).astype(int)     # more unpaid
    + (df_target["base_ontime_rate_chg_37_8_vs_97_38"] < 0).astype(int) # on-time rate falling
    + df_target["base_gone_silent_flag"]                                  # stopped paying
)   # range 0–5

# Recovery signal: all dimensions improving recently
df_target["base_recovery_score"] = (
    (df_target["base_max_dpd_chg_37_8_vs_97_38"] < 0).astype(int)       # DPD improving
    + (df_target["base_pay_freq_chg_37_8_vs_97_38"] > 0).astype(int)    # paying more often
    + (df_target["base_ontime_rate_chg_37_8_vs_97_38"] > 0).astype(int) # on-time improving
    + df_target["base_steady_payer_flag"]                                 # consistently paying
)   # range 0–4

# Net trajectory: positive = recovering, negative = deteriorating
df_target["base_net_trajectory"] = (
    df_target["base_recovery_score"] - df_target["base_deterioration_score"]
)

# ── Invoice amount trends ─────────────────────────────────────────────
# Is the borrower invoiced less recently? (could mean loan nearing end)
df_target["base_invoiced_amt_chg_37_8_vs_97_38"]   = safe_diff(df_target["base_invoiced_amt_37_8"],  df_target["base_invoiced_amt_97_38"])
df_target["base_invoiced_amt_chg_97_38_vs_187_98"] = safe_diff(df_target["base_invoiced_amt_97_38"], df_target["base_invoiced_amt_187_98"])

# Invoiced amount concentration (how much of total is recent?)
total_invoiced = (
    df_target["base_invoiced_amt_372_188"]
    + df_target["base_invoiced_amt_187_98"]
    + df_target["base_invoiced_amt_97_38"]
    + df_target["base_invoiced_amt_37_8"]
)
df_target["base_recent_invoiced_concentration"] = safe_div(
    df_target["base_invoiced_amt_37_8"] + df_target["base_invoiced_amt_97_38"],
    total_invoiced
)

print(f"New features added: {len([c for c in df_target.columns if 'chg' in c or 'ratio' in c or 'trajectory' in c or 'acceleration' in c or 'severity' in c or 'deterioration' in c or 'recovery' in c])}")

In [ ]:
df_target

In [ ]:
df_target.shape

In [ ]:
def build_card_behavioral_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build card add/remove behavioral features from raw wallet activity columns.
    """
    out = df.copy()

    # ── base columns ──────────────────────────────────────────────────────────
    a1w = out["card_add_cnt_w_1w"].fillna(0)
    a1m = out["card_add_cnt_w_1m"].fillna(0)
    a3m = out["card_add_cnt_w_3m"].fillna(0)
    a6m = out["card_add_cnt_w_6m"].fillna(0)

    r1w = out["card_remove_cnt_w_1w"].fillna(0)
    r1m = out["card_remove_cnt_w_1m"].fillna(0)
    r3m = out["card_remove_cnt_w_3m"].fillna(0)
    r6m = out["card_remove_cnt_w_6m"].fillna(0)

    # ─────────────────────────────────────────────────────────────────────────
    # ADD MOMENTUM
    # ─────────────────────────────────────────────────────────────────────────
    out["card_add_momentum_1m_3m"] = np.where(a3m == 0, 0, (a1m - a3m) / a3m)
    out["card_add_momentum_1m_6m"] = np.where(a6m == 0, 0, (a1m - a6m) / a6m)
    out["card_add_momentum_3m_6m"] = np.where(a6m == 0, 0, (a3m - a6m) / a6m)
    out["card_add_acceleration"] = a1m - 2 * a3m + a6m

    # ─────────────────────────────────────────────────────────────────────────
    # REMOVE MOMENTUM
    # ─────────────────────────────────────────────────────────────────────────
    out["card_remove_momentum_1m_3m"] = np.where(r3m == 0, 0, (r1m - r3m) / r3m)
    out["card_remove_momentum_1m_6m"] = np.where(r6m == 0, 0, (r1m - r6m) / r6m)
    out["card_remove_momentum_3m_6m"] = np.where(r6m == 0, 0, (r3m - r6m) / r6m)
    out["card_remove_acceleration"] = r1m - 2 * r3m + r6m

    # ─────────────────────────────────────────────────────────────────────────
    # NET ACTIVITY  (adds - removes per window)
    # ─────────────────────────────────────────────────────────────────────────
    out["card_net_activity_1w"] = a1w - r1w
    out["card_net_activity_1m"] = a1m - r1m
    out["card_net_activity_3m"] = a3m - r3m
    out["card_net_activity_6m"] = a6m - r6m

    # ─────────────────────────────────────────────────────────────────────────
    # CHURN RATIO  (removes / adds — how much of what was added is being pulled)
    # ─────────────────────────────────────────────────────────────────────────
    out["card_churn_ratio_1m"] = np.where(a1m == 0, 0, r1m / a1m)
    out["card_churn_ratio_3m"] = np.where(a3m == 0, 0, r3m / a3m)
    out["card_churn_ratio_6m"] = np.where(a6m == 0, 0, r6m / a6m)

    # ─────────────────────────────────────────────────────────────────────────
    # RECENCY SHARE  (how concentrated is activity in the most recent window)
    # ─────────────────────────────────────────────────────────────────────────
    out["card_add_recent_share"] = np.where(a6m == 0, 0, a1m / a6m)
    out["card_remove_recent_share"] = np.where(r6m == 0, 0, r1m / r6m)

    # ─────────────────────────────────────────────────────────────────────────
    # VOLATILITY  (how uneven is the add/remove pattern across windows)
    # ─────────────────────────────────────────────────────────────────────────
    _add_sum = a1m + a3m + a6m
    out["card_add_volatility"] = np.where(
        _add_sum == 0, 0, (np.abs(a1m - a3m) + np.abs(a3m - a6m)) / _add_sum
    )

    _rem_sum = r1m + r3m + r6m
    out["card_remove_volatility"] = np.where(
        _rem_sum == 0, 0, (np.abs(r1m - r3m) + np.abs(r3m - r6m)) / _rem_sum
    )

    # ─────────────────────────────────────────────────────────────────────────
    # FLAGS
    # ─────────────────────────────────────────────────────────────────────────
    # active card management in last week
    out["card_active_flag"] = ((a1w > 0) | (r1w > 0)).astype(int)

    # removed cards recently but added none → distress signal
    out["card_removal_only_flag"] = ((r1m > 0) & (a1m == 0)).astype(int)

    # removed everything they added in same window
    out["card_full_churn_flag"] = ((r1m >= a1m) & (a1m > 0)).astype(int)

    # sudden spike: this month's removes > 2x the 3-month average rate
    out["card_remove_spike_flag"] = (
        r1m > 2 * (r3m / 3).replace(0, np.nan).fillna(0)
    ).astype(int)

    # no card adds in 6 months → disengaged from wallet
    out["card_dormant_flag"] = (a6m == 0).astype(int)

    # escalating removal: removes accelerating into most recent period
    out["card_remove_escalation_flag"] = ((r1m > r3m) & (r3m > r6m / 2)).astype(int)

    # ─────────────────────────────────────────────────────────────────────────
    # COMPOSITE SCORE  (wallet stability — higher = more stable)
    # ─────────────────────────────────────────────────────────────────────────
    # component 1: low churn ratio in 1m (inverted, capped at 1)
    _churn_component = 1 - np.clip(np.where(a1m == 0, 0, r1m / a1m), 0, 1)
    # component 2: net activity is non-negative
    _net_component = np.where(out["card_net_activity_1m"] >= 0, 1, 0)
    # component 3: no recent spike
    _spike_component = 1 - out["card_remove_spike_flag"]

    out["card_wallet_stability_score"] = (
        _churn_component + _net_component + _spike_component
    ) / 3

    return out


df_target = build_card_behavioral_features(df_target)

In [ ]:
# ── App behavior change rate features ────────────────────────────────

# Transaction count momentum (positive = more active recently)
df_target["app_trx_chg_l7_vs_l30"]  = safe_div(df_target["app_trx_l7d"]  * 4, df_target["app_trx_l30d"])   # annualized to same scale
df_target["app_trx_chg_l30_vs_l60"] = safe_div(df_target["app_trx_l30d"] * 2, df_target["app_trx_l60d"])
df_target["app_trx_chg_l30_vs_l90"] = safe_div(df_target["app_trx_l30d"] * 3, df_target["app_trx_l90d"])
df_target["app_trx_chg_l60_vs_l90"] = safe_div(df_target["app_trx_l60d"] * 1.5, df_target["app_trx_l90d"])

# Raw diff (simpler signal — did absolute trx count drop?)
df_target["app_trx_diff_l7_vs_l30"]  = df_target["app_trx_l7d"]  - (df_target["app_trx_l30d"]  / 4)
df_target["app_trx_diff_l30_vs_l60"] = df_target["app_trx_l30d"] - (df_target["app_trx_l60d"]  / 2)
df_target["app_trx_diff_l30_vs_l90"] = df_target["app_trx_l30d"] - (df_target["app_trx_l90d"]  / 3)

# ── Avg transaction amount trend ──────────────────────────────────────
# Positive = spending more per transaction recently (financial health signal)
df_target["app_amt_chg_l7_vs_l30"]  = safe_diff(df_target["app_avg_trx_amount_l7d"],  df_target["app_avg_trx_amount_l30d"])
df_target["app_amt_chg_l30_vs_l60"] = safe_diff(df_target["app_avg_trx_amount_l30d"], df_target["app_avg_trx_amount_l60d"])
df_target["app_amt_chg_l30_vs_l90"] = safe_diff(df_target["app_avg_trx_amount_l30d"], df_target["app_avg_trx_amount_l90d"])
df_target["app_amt_chg_l60_vs_l90"] = safe_diff(df_target["app_avg_trx_amount_l60d"], df_target["app_avg_trx_amount_l90d"])

# Ratio version (>1 = spending more per trx recently)
df_target["app_amt_ratio_l7_vs_l90"]  = safe_div(df_target["app_avg_trx_amount_l7d"],  df_target["app_avg_trx_amount_l90d"])
df_target["app_amt_ratio_l30_vs_l90"] = safe_div(df_target["app_avg_trx_amount_l30d"], df_target["app_avg_trx_amount_l90d"])

# ── Session engagement trend ──────────────────────────────────────────
# Positive = logging in more frequently recently
df_target["app_sessions_chg_l7_vs_l30"]  = safe_diff(df_target["app_sessions_l7d"]  * 4, df_target["app_sessions_l30d"])
df_target["app_sessions_chg_l30_vs_l60"] = safe_diff(df_target["app_sessions_l30d"] * 2, df_target["app_sessions_l60d"])
df_target["app_sessions_chg_l30_vs_l90"] = safe_diff(df_target["app_sessions_l30d"] * 3, df_target["app_sessions_l90d"])

# Sessions per transaction — high = browsing without paying (disengaged)
df_target["app_sessions_per_trx_l30"] = safe_div(df_target["app_sessions_l30d"], df_target["app_trx_l30d"])
df_target["app_sessions_per_trx_l90"] = safe_div(df_target["app_sessions_l90d"], df_target["app_trx_l90d"])
df_target["app_sessions_per_trx_chg"] = safe_diff(
    df_target["app_sessions_per_trx_l30"],
    df_target["app_sessions_per_trx_l90"]
)

# ── Notification read rate trend ──────────────────────────────────────
# Ignoring notifications = disengaging from repayment reminders
df_target["app_noti_read_rate_l7"]  = safe_div(df_target["app_read_noti_l7d"],  df_target["app_sessions_l7d"])
df_target["app_noti_read_rate_l30"] = safe_div(df_target["app_read_noti_l30d"], df_target["app_sessions_l30d"])
df_target["app_noti_read_rate_l60"] = safe_div(df_target["app_read_noti_l60d"], df_target["app_sessions_l60d"])
df_target["app_noti_read_rate_l90"] = safe_div(df_target["app_read_noti_l90d"], df_target["app_sessions_l90d"])

df_target["app_noti_read_rate_chg_l7_vs_l30"]  = safe_diff(df_target["app_noti_read_rate_l7"],  df_target["app_noti_read_rate_l30"])
df_target["app_noti_read_rate_chg_l30_vs_l90"] = safe_diff(df_target["app_noti_read_rate_l30"], df_target["app_noti_read_rate_l90"])

# Raw noti read diff (annualized to same period)
df_target["app_noti_chg_l7_vs_l30"]  = safe_diff(df_target["app_read_noti_l7d"]  * 4, df_target["app_read_noti_l30d"])
df_target["app_noti_chg_l30_vs_l90"] = safe_diff(df_target["app_read_noti_l30d"] * 3, df_target["app_read_noti_l90d"])

# ── Composite engagement score ────────────────────────────────────────
# Combines trx + session + noti signals into one disengagement flag
# Higher = more signs of disengagement recently
df_target["app_disengagement_score"] = (
    (df_target["app_trx_diff_l30_vs_l90"] < 0).astype(int)           # fewer trx recently
    + (df_target["app_sessions_chg_l30_vs_l90"] < 0).astype(int)     # fewer sessions
    + (df_target["app_noti_read_rate_chg_l30_vs_l90"] < 0).astype(int)  # ignoring notis more
    + (df_target["app_trx_recency_l90d"] > 30).astype(int)            # last trx was long ago
)   # range 0–4

# Engagement momentum: recent 30d vs baseline 90d (>1 = re-engaging)
df_target["app_engagement_momentum"] = safe_div(
    df_target["app_sessions_l30d"] * 3 + df_target["app_trx_l30d"] * 3,
    df_target["app_sessions_l90d"] + df_target["app_trx_l90d"] + 1
)

print(f"App features added: {len([c for c in df_target.columns if c.startswith('app_') and any(x in c for x in ['chg','ratio','rate','diff','score','momentum','per_trx'])])}")

In [ ]:
def build_cdr_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    c30 = df["cdr_cdr_cnt_l30"].fillna(0)
    c60 = df["cdr_cdr_cnt_l60"].fillna(0)
    c90 = df["cdr_cdr_cnt_l90"].fillna(0)
    cb90 = df["cdr_cdr_count_before90"].fillna(0)

    n30 = df["cdr_number_change_cnt_l30d"].fillna(0)
    n60 = df["cdr_number_change_cnt_l60d"].fillna(0)
    n90 = df["cdr_number_change_cnt_l90d"].fillna(0)
    ntot = df["cdr_number_change_cnt"].fillna(0)

    def safe_div(a, b):
        return a.div(b.replace(0, np.nan)).fillna(0)

    # ── CDR momentum (how fast call records are growing/shrinking) ──
    df["cdr_momentum_30_60"] = safe_div(c30 - c60, c60)  # L30 vs L60 base
    df["cdr_momentum_30_90"] = safe_div(c30 - c90, c90)  # L30 vs L90 base
    df["cdr_momentum_60_90"] = safe_div(c60 - c90, c90)  # L60 vs L90 base
    df["cdr_momentum_vs_old"] = safe_div(
        c90 - cb90, cb90
    )  # recent 90d vs older history

    # ── CDR change rate (absolute period-over-period rate) ──
    # --------------------------------------------------
    # 30 vs 60
    # --------------------------------------------------

    change_30_60 = safe_div(c30 - c60, c60)

    df["cdr_change_rate_30_60"] = change_30_60

    df["cdr_extreme_drop_30_60"] = (change_30_60 < -0.5).astype(int)

    df["cdr_normal_range_30_60"] = (
        (change_30_60 >= -0.5) & (change_30_60 <= -0.05)
    ).astype(int)

    df["cdr_growth_30_60"] = (change_30_60 > -0.05).astype(int)

    # --------------------------------------------------
    # 60 vs 90
    # --------------------------------------------------

    change_60_90 = safe_div(c60 - c90, c90)

    df["cdr_change_rate_60_90"] = change_60_90

    df["cdr_extreme_drop_60_90"] = (change_60_90 < -0.5).astype(int)

    df["cdr_normal_range_60_90"] = (
        (change_60_90 >= -0.5) & (change_60_90 <= -0.05)
    ).astype(int)

    df["cdr_growth_60_90"] = (change_60_90 > -0.05).astype(int)

    # --------------------------------------------------
    # 30 vs 90
    # --------------------------------------------------

    change_30_90 = safe_div(c30 - c90, c90)

    df["cdr_change_rate_30_90"] = change_30_90

    df["cdr_extreme_drop_30_90"] = (change_30_90 < -0.5).astype(int)

    df["cdr_normal_range_30_90"] = (
        (change_30_90 >= -0.5) & (change_30_90 <= -0.05)
    ).astype(int)

    df["cdr_growth_30_90"] = (change_30_90 > -0.05).astype(int)

    # ── CDR acceleration (second derivative — is growth itself speeding up?) ──
    df["cdr_acceleration"] = c30 - 2 * c60 + c90

    # ── CDR drop flags ──
    df["cdr_drop_30_flag"] = ((c30 < c60) & (c60 > 0)).astype(int)
    df["cdr_drop_60_flag"] = ((c60 < c90) & (c90 > 0)).astype(int)
    df["cdr_sudden_drop_flag"] = ((c30 == 0) & (c60 > 0)).astype(
        int
    )  # went silent last 30d
    df["cdr_gone_silent_flag"] = ((c30 == 0) & (c90 > 0)).astype(
        int
    )  # no calls in 30d but had in 90d

    # ── CDR concentration (how much of recent 90d activity is in last 30d) ──
    df["cdr_recent_share"] = safe_div(c30, c90)  # >0.5 = front-loaded
    df["cdr_consistency_index"] = safe_div(c30, c90)  # alias for scorecard naming

    # ── CDR vs old history ratio (recovery or escalation vs pre-90d) ──
    df["cdr_vs_history_ratio"] = safe_div(c90, cb90 + c90)

    # ── NUMBER CHANGE momentum ──
    df["numchg_momentum_30_60"] = safe_div(n30 - n60, n60)
    df["numchg_momentum_30_90"] = safe_div(n30 - n90, n90)
    df["numchg_momentum_60_90"] = safe_div(n60 - n90, n90)

    # ── NUMBER CHANGE acceleration ──
    df["numchg_acceleration"] = n30 - 2 * n60 + n90

    # ── NUMBER CHANGE rate vs total lifetime ──
    df["numchg_recent_share"] = safe_div(n30, ntot)  # recent changes as % of all-time
    df["numchg_90d_share"] = safe_div(n90, ntot)

    # ── NUMBER CHANGE drop flags ──
    df["numchg_spike_30_flag"] = ((n30 > n60) & (n60 > 0)).astype(
        int
    )  # accelerating changes = instability
    df["numchg_recent_flag"] = (n30 > 0).astype(int)  # any change in last 30d
    df["numchg_frequent_flag"] = (safe_div(n90, c90 + 1) > 0.3).astype(
        int
    )  # changes relative to call volume

    # ── COMBINED signals (CDR + number change together) ──
    # High CDR + frequent number changes = hard to reach / evasion signal
    df["cdr_numchg_risk_score"] = (
        safe_div(c30, c90 + 1) * 0.5  # CDR recent concentration
        + safe_div(n30, ntot + 1) * 0.5  # number change recency
    )
    df["contact_instability_flag"] = (
        (df["numchg_recent_flag"] == 1) & (df["cdr_drop_30_flag"] == 1)
    ).astype(
        int
    )  # changed number AND fewer calls this month = evasion

    return df


df_target = build_cdr_features(df_target)

In [ ]:
df_target.shape

In [ ]:
import numpy as np

EPS = 1e-6


def build_mobile_features(df):
    cols_12m = [
        "mobile_avg_inv_amt_12m",
        "mobile_max_inv_amt_12m",
        "mobile_median_pay_days_12m",
        "mobile_invoice_cnt_12m",
        "mobile_invoice_ontime_cnt_12m",
    ]

    cols_3m = [
        "mobile_avg_inv_amt_3m",
        "mobile_max_inv_amt_3m",
        "mobile_median_pay_days_3m",
        "mobile_invoice_cnt_3m",
        "mobile_invoice_ontime_cnt_3m",
    ]

    cols_1m = [
        "mobile_avg_inv_amt_1m",
        "mobile_max_inv_amt_1m",
        "mobile_median_pay_days_1m",
        "mobile_invoice_cnt_1m",
        "mobile_invoice_ontime_cnt_1m",
    ]

    base_names = [
        "avg_inv_amt",
        "max_inv_amt",
        "median_pay_days",
        "invoice_cnt",
        "invoice_ontime_cnt",
    ]

    for i, base in enumerate(base_names):
        c12, c3, c1 = cols_12m[i], cols_3m[i], cols_1m[i]

        # -----------------------
        # CHANGE RATES
        # -----------------------
        df[f"{base}_chg_1m_vs_12m"] = (df[c1] - df[c12]) / (df[c12] + EPS)
        df[f"{base}_chg_3m_vs_12m"] = (df[c3] - df[c12]) / (df[c12] + EPS)
        df[f"{base}_chg_1m_vs_3m"] = (df[c1] - df[c3]) / (df[c3] + EPS)

        # -----------------------
        # MOMENTUM
        # -----------------------
        df[f"{base}_momentum_1m_3m"] = df[c1] - df[c3]
        df[f"{base}_momentum_3m_12m"] = df[c3] - df[c12]

        # -----------------------
        # ACCELERATION
        # -----------------------
        df[f"{base}_acceleration"] = (df[c1] - df[c3]) - (df[c3] - df[c12])

        # -----------------------
        # RATIO STABILITY (optional but useful)
        # -----------------------
        df[f"{base}_ratio_stability"] = (df[c1] + EPS) / (df[c12] + EPS)

    return df


df_target = build_mobile_features(df_target)
print(df_target.shape)

In [ ]:
import numpy as np

EPS = 1e-6


def build_uloan_features(df):

    # ----------------------------
    # LOAN MOMENTUM (AMOUNT)
    # ----------------------------
    df["uloan_loan_amt_momentum_30d_90d"] = df["uloan_loan_amt_l30d"] - df[
        "uloan_loan_amt_l90d"
    ] * (30 / 90)

    df["uloan_loan_amt_momentum_7d_30d"] = df["uloan_loan_amt_l7d"] - df[
        "uloan_loan_amt_l30d"
    ] * (7 / 30)

    # change rates
    df["uloan_loan_amt_chg_30d_vs_90d"] = (
        df["uloan_loan_amt_l30d"] - df["uloan_loan_amt_l90d"]
    ) / (df["uloan_loan_amt_l90d"] + EPS)

    df["uloan_loan_amt_chg_7d_vs_30d"] = (
        df["uloan_loan_amt_l7d"] - df["uloan_loan_amt_l30d"]
    ) / (df["uloan_loan_amt_l30d"] + EPS)

    # acceleration
    df["uloan_loan_amt_acceleration"] = (
        df["uloan_loan_amt_l7d"]
        - 2 * df["uloan_loan_amt_l30d"]
        + df["uloan_loan_amt_l90d"]
    )

    # spike detection
    df["uloan_loan_spike_7d_ratio"] = df["uloan_loan_amt_l7d"] / (
        df["uloan_loan_amt_l90d"] + EPS
    )

    df["uloan_loan_concentration_7d"] = df["uloan_loan_amt_l7d"] / (
        df["uloan_loan_amt_l30d"] + EPS
    )

    # ----------------------------
    # REPAYMENT MOMENTUM
    # ----------------------------
    df["uloan_repay_momentum_30d_90d"] = df["uloan_repay_amt_l30d"] - df[
        "uloan_repay_amt_l90d"
    ] * (30 / 90)

    df["uloan_repay_momentum_7d_30d"] = df["uloan_repay_amt_l7d"] - df[
        "uloan_repay_amt_l30d"
    ] * (7 / 30)

    df["uloan_repay_chg_30d_vs_90d"] = (
        df["uloan_repay_amt_l30d"] - df["uloan_repay_amt_l90d"]
    ) / (df["uloan_repay_amt_l90d"] + EPS)

    df["uloan_repay_chg_7d_vs_30d"] = (
        df["uloan_repay_amt_l7d"] - df["uloan_repay_amt_l30d"]
    ) / (df["uloan_repay_amt_l30d"] + EPS)

    df["uloan_repay_acceleration"] = (
        df["uloan_repay_amt_l7d"]
        - 2 * df["uloan_repay_amt_l30d"]
        + df["uloan_repay_amt_l90d"]
    )

    # repayment efficiency
    df["uloan_repay_to_loan_ratio_90d"] = df["uloan_repay_amt_l90d"] / (
        df["uloan_loan_amt_l90d"] + EPS
    )

    df["uloan_repay_to_loan_ratio_30d"] = df["uloan_repay_amt_l30d"] / (
        df["uloan_loan_amt_l30d"] + EPS
    )

    df["uloan_repay_to_loan_ratio_7d"] = df["uloan_repay_amt_l7d"] / (
        df["uloan_loan_amt_l7d"] + EPS
    )

    df["uloan_repay_ratio_momentum"] = (
        df["uloan_repay_to_loan_ratio_7d"] - df["uloan_repay_to_loan_ratio_30d"]
    )

    # ----------------------------
    # RECENCY
    # ----------------------------
    df["uloan_last_loan_recency_log"] = np.log1p(df["uloan_last_loan_recency"])
    df["uloan_last_repay_recency_log"] = np.log1p(df["uloan_last_repay_recency"])

    df["uloan_activity_decay"] = (
        df["uloan_last_loan_recency"] - df["uloan_last_repay_recency"]
    )

    # ----------------------------
    # BEHAVIOR FLAGS
    # ----------------------------
    df["uloan_recent_borrowing_spike"] = (
        df["uloan_loan_amt_l7d"] > df["uloan_loan_amt_l30d"]
    ).astype(int)

    df["uloan_recent_repay_drop"] = (
        df["uloan_repay_amt_l7d"] < df["uloan_repay_amt_l30d"] * 0.5
    ).astype(int)

    return df


df_target = build_uloan_features(df_target)
print(df_target.shape)

In [ ]:
df_target.describe()

In [ ]:
cols_with_nulls = df_target.columns[df_target.isnull().sum() > 0]
cols_with_nulls

In [ ]:
num_cols = df_target.select_dtypes(include="number").columns.tolist()
cat_cols = df_target.select_dtypes(include=["object", "string"]).columns.tolist()

In [ ]:
df_target.loc[:, ~df_target.columns.isin(num_cols)]

In [ ]:
df_target.drop(columns=["gprod_type"], inplace=True)

In [ ]:
print(len(num_cols))
print(len(cat_cols))
print(len(df_target.columns.tolist()))

In [ ]:
cat_cols

In [ ]:
df_target.shape

### Split with date

In [ ]:
df_target["date"] = pd.to_datetime(df_target["base_date"]).dt.strftime("%Y%m")
# df_target.drop(columns=['base_date'], axis=1, inplace=True )


# Out-of-time split windows (YYYYMM) — adjust to your data
TRAIN_START, TRAIN_END = "202301", "202310"   # [start, end)
VALID4_START, VALID4_END = "202310", "202311"
VALID5_START, VALID5_END = "202311", "202312"  # inclusive end


def classify_date(date):
    if TRAIN_START <= date < TRAIN_END:
        return "train"
    if VALID4_START <= date < VALID4_END:
        return "valid4"
    elif VALID5_START <= date <= VALID5_END:
        return "valid5"


df_target["data_type"] = df_target["date"].apply(classify_date)

# df_target.drop(columns=['date', 'base_date'], axis=1, inplace=True )

In [ ]:
df_target.head()

In [ ]:
df_target.drop(columns=["date"], inplace=True)
df_target.head()

In [ ]:
df_test4 = df_target[df_target["data_type"] == "valid4"]
df_test5 = df_target[df_target["data_type"] == "valid5"]
df_train = df_target[df_target["data_type"] == "train"]

In [ ]:
df_target.describe()

In [ ]:
df_train.describe()

In [ ]:
print(df_test4.shape)
print(df_test5.shape)
print(df_train.shape)
print(df_target.shape)

#### Post Cleaning

In [ ]:
import pandas as pd
import numpy as np
import scorecardpy as sc
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ─────────────────────────────────────────────────────────────
# STEP 2A — Drop columns with too much missing
# ─────────────────────────────────────────────────────────────
MISSING_THRESHOLD = 0.50  # drop if >50% null

missing_rate = df_train.isnull().mean()
drop_missing = missing_rate[missing_rate > MISSING_THRESHOLD].index.tolist()
print(f"Dropping {len(drop_missing)} cols for high missingness:\n", drop_missing)
df_train = df_train.drop(columns=drop_missing)

# ─────────────────────────────────────────────────────────────
# STEP 2B — Fix known edge-case artifacts from your SQL
# ─────────────────────────────────────────────────────────────
# Your ratio features (momentum, decay, growth) can produce:
#   - Inf      → from NULLIF that still slips through in Python
#   - -Inf
#   - NaN      → 0/0 edge cases
#   - Extreme values like 999 (trx_l30d=30, trx_l90d=0.03)

ratio_cols = [
    c
    for c in df_train.columns
    if any(
        x in c
        for x in [
            "momentum",
            "decay",
            "growth",
            "ratio",
            "acceleration",
            "volatility",
            "consistency",
            "stability",
            "score",
            "rate",
            "share",
        ]
    )
]

# Replace Inf/-Inf with NaN first, then fill
df_train[ratio_cols] = df_train[ratio_cols].replace([np.inf, -np.inf], np.nan)

# Fill NaN in ratio/derived cols with 0 (matches your SQL NVL logic)
df_train[ratio_cols] = df_train[ratio_cols].fillna(0)

# Fill remaining numeric NaNs with median
num_cols = df_train.select_dtypes(include="number").columns.difference(["bad"])
for col in num_cols:
    if df_train[col].isnull().any():
        df_train[col] = df_train[col].fillna(df_train[col].median())

# ─────────────────────────────────────────────────────────────
# STEP 2C — Outlier removal using IQR capping (Winsorization)
# ─────────────────────────────────────────────────────────────
# Cap (not drop) outliers — important for scorecard stability
# Do NOT cap binary flags or the target

flag_cols = [c for c in df_train.columns if c.endswith("_flag")]
skip_cap = flag_cols + ["bad"]


def iqr_cap(series, lower_pct=0.01, upper_pct=0.99):
    """Cap at 1st/99th percentile. More conservative than IQR for skewed ratios."""
    lo = series.quantile(lower_pct)
    hi = series.quantile(upper_pct)
    return series.clip(lower=lo, upper=hi)


cap_cols = [c for c in num_cols if c not in skip_cap]
for col in cap_cols:
    df_train[col] = iqr_cap(df_train[col])

# Inspect how many rows were affected
print("Capping applied to", len(cap_cols), "columns")

# ─────────────────────────────────────────────────────────────
# STEP 2D — Drop near-constant columns (manual safety net
#            before var_filter, catches post-cap flatlines)
# ─────────────────────────────────────────────────────────────
IDENTICAL_THRESHOLD = 0.98

for col in num_cols:
    if "avg" in col or "max" in col or "median" in col:
        continue  # skip these, they are expected to be constant sometimes
    top_freq = df_train[col].value_counts(normalize=True).iloc[0]
    if top_freq > IDENTICAL_THRESHOLD:
        print(f"Near-constant, dropping: {col}  ({top_freq:.1%} same value)")
        df_train = df_train.drop(columns=[col])

In [ ]:
# ─────────────────────────────────────────────────────────────
# PSI FILTER — same logic as monthly PSI report
# Base month vs every future month
# ─────────────────────────────────────────────────────────────
out_dir = OUT_DIR

DATE_COL = "base_date"
PSI_DROP_THRESHOLD = 0.25


def psi_score(expected, actual, n_bins=10):
    mn = min(expected.min(), actual.min())
    mx = max(expected.max(), actual.max())

    if mn == mx:
        return 0.0

    edges = np.linspace(mn, mx, n_bins + 1)

    e_pct = np.histogram(expected, bins=edges)[0] / max(len(expected), 1)
    a_pct = np.histogram(actual, bins=edges)[0] / max(len(actual), 1)

    e_pct = np.where(e_pct == 0, 1e-4, e_pct)
    a_pct = np.where(a_pct == 0, 1e-4, a_pct)

    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))


# ------------------------------------------------------------
# Create month slices
# ------------------------------------------------------------

df_train[DATE_COL] = pd.to_datetime(df_train[DATE_COL])
df_train["month"] = df_train[DATE_COL].dt.to_period("M").astype(str)

months_sorted = sorted(df_train["month"].unique())
base_month = months_sorted[0]

print(
    f"PSI filter: base={base_month}, "
    f"{len(months_sorted)} months, "
    f"threshold={PSI_DROP_THRESHOLD}"
)

# ------------------------------------------------------------
# Feature list
# ------------------------------------------------------------

feat_cols = df_train.select_dtypes(include="number").columns.tolist()

feat_cols = [c for c in feat_cols if c not in ["bad", "loan_tenure", "base_s360", "base_inquiry", "loan_principal_amt", "base_ontime_rate_chg_37_8_vs_97_38"]]

# ------------------------------------------------------------
# Compute max PSI against base month
# ------------------------------------------------------------

max_base_psi = {}

for feat in feat_cols:

    base_vals = df_train.loc[df_train["month"] == base_month, feat].dropna().values

    if len(base_vals) == 0:
        max_base_psi[feat] = np.nan
        continue

    psis = []

    for month in months_sorted[1:]:

        month_vals = df_train.loc[df_train["month"] == month, feat].dropna().values

        if len(month_vals) == 0:
            continue

        psis.append(psi_score(base_vals, month_vals))

    max_base_psi[feat] = max(psis) if psis else 0.0

# ------------------------------------------------------------
# PSI summary
# ------------------------------------------------------------

psi_df = pd.DataFrame.from_dict(
    max_base_psi, orient="index", columns=["max_base_psi"]
).sort_values("max_base_psi", ascending=False)

drop_psi = psi_df[psi_df["max_base_psi"] >= PSI_DROP_THRESHOLD].index.tolist()

print(f"\n✗ Dropping {len(drop_psi)} unstable features")

if len(drop_psi):
    print(psi_df[psi_df["max_base_psi"] >= PSI_DROP_THRESHOLD].to_string())

print(f"\n✓ Keeping {len(feat_cols) - len(drop_psi)} stable features")

# ------------------------------------------------------------
# Drop unstable features
# ------------------------------------------------------------

df_train = df_train.drop(columns=drop_psi + ["month"], errors="ignore")

psi_df.to_csv(f"{out_dir}/psi_filter_log.csv")

In [ ]:
monthly_bad_rate = (
    df_train
    .groupby(df_train['base_date'].dt.to_period('M'))['bad']
    .mean()
    .reset_index()
)

monthly_bad_rate['base_date'] = monthly_bad_rate['base_date'].astype(str)

print(monthly_bad_rate)

### Training

In [ ]:
segments = {
    "new_loans": df_train[(df_train["loan_tenure"] <= 90)],
    "old_loans": df_train[df_train["loan_tenure"] >= 91],
    "all_loans": df_train,
}

In [ ]:
def cutoff_analysis(score_df, y, seg_name, seg_dir, score_col="score"):
    df = score_df[[score_col]].copy()
    df["bad"] = y.values

    # Auto-detect score range from actual data
    s_min = int(df[score_col].min() // 10) * 10
    s_max = int(df[score_col].max() // 10) * 10 + 10

    print
    rows = []
    for cutoff in range(s_min, s_max, 50):
        above = df[df[score_col] >= cutoff]
        below = df[df[score_col] < cutoff]
        rows.append(
            {
                "cutoff": cutoff,
                "n_above": len(above),
                "n_below": len(below),
                "pct_above": round(len(above) / len(df), 4),
                "bad_rate_above": (
                    round(above["bad"].mean(), 4) if len(above) else np.nan
                ),
                "bad_rate_below": (
                    round(below["bad"].mean(), 4) if len(below) else np.nan
                ),
                "bads_captured_below": (
                    round(below["bad"].sum() / df["bad"].sum(), 4)
                    if df["bad"].sum()
                    else np.nan
                ),
            }
        )

    cutoff_df = pd.DataFrame(rows)
    cutoff_df.to_csv(f"{seg_dir}/{seg_name}_cutoff_analysis.csv", index=False)

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax2 = ax1.twinx()
    ax1.plot(
        cutoff_df["cutoff"],
        cutoff_df["pct_above"],
        color="steelblue",
        label="% Population above cutoff",
    )
    ax1.plot(
        cutoff_df["cutoff"],
        cutoff_df["bads_captured_below"],
        color="orange",
        label="% Bads captured below cutoff",
    )
    ax2.plot(
        cutoff_df["cutoff"],
        cutoff_df["bad_rate_above"],
        color="green",
        linestyle="--",
        label="Bad rate above cutoff",
    )
    ax2.plot(
        cutoff_df["cutoff"],
        cutoff_df["bad_rate_below"],
        color="crimson",
        linestyle="--",
        label="Bad rate below cutoff",
    )
    ax1.set_xlabel("Score Cutoff")
    ax1.set_ylabel("Population / Bad Capture %")
    ax2.set_ylabel("Bad Rate")
    ax1.legend(loc="upper left")
    ax2.legend(loc="upper right")
    ax1.set_title(f"{seg_name} — Cutoff analysis")
    plt.tight_layout()
    plt.savefig(f"{seg_dir}/{seg_name}_cutoff_analysis.png", dpi=150)
    plt.close()

    return cutoff_df


def save_woebin_plots(final_bins, seg_name, seg_dir):
    """sc.woebin_plot returns a dict of {var: Figure} — save each one."""
    plots = sc.woebin_plot(final_bins)
    for var, fig in plots.items():
        fig.savefig(f"{seg_dir}/{seg_name}_woe_{var}.png", dpi=150, bbox_inches="tight")
        plt.close(fig)


def score_distribution_plot(
    train_score, test_score, y_train, y_test, seg_name, seg_dir
):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, (score_df, y, label) in zip(
        axes, [(train_score, y_train, "train"), (test_score, y_test, "test")]
    ):
        plot_df = pd.DataFrame({"score": score_df["score"].values, "bad": y.values})
        sns.histplot(data=plot_df, x="score", hue="bad", bins=30, ax=ax)
        ax.set_title(f"{seg_name} — {label} score distribution")
    plt.tight_layout()
    plt.savefig(f"{seg_dir}/{seg_name}_score_distribution.png", dpi=150)
    plt.close()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import math


def plot_woe_bins(
    bins: dict,
    seg_name: str = "",
    save_dir: str = None,
    cols: int = 3,
    figsize_per: tuple = (4, 3.5),
):
    """
    Clean WOE bin visualization replacing sc.woebin_plot().
    - Rotated x-axis labels (no overlap)
    - WOE bar + event rate line on dual axes
    - Bin count annotated on each bar
    """
    n = len(bins)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(
        rows, cols, figsize=(figsize_per[0] * cols, figsize_per[1] * rows)
    )
    axes = np.array(axes).flatten()

    for idx, (var, df) in enumerate(bins.items()):
        ax1 = axes[idx]

        # ── Data ────────────────────────────────────────────────────
        labels = df["bin"].astype(str).tolist()
        woe = df["woe"].tolist()
        bad_rt = df["badprob"].tolist()  # event rate per bin
        counts = df["count"].tolist()
        iv_val = df["total_iv"].iloc[0]

        x = np.arange(len(labels))
        bar_colors = ["#c0392b" if w < 0 else "#2980b9" for w in woe]

        # ── Bar: WOE ────────────────────────────────────────────────
        bars = ax1.bar(x, woe, color=bar_colors, alpha=0.75, width=0.55, zorder=2)
        ax1.axhline(0, color="#555", linewidth=0.8, linestyle="--")
        ax1.set_ylabel("WOE", fontsize=8, color="#333")
        ax1.tick_params(axis="y", labelsize=7)
        ax1.set_xticks(x)
        ax1.set_xticklabels(labels, rotation=40, ha="right", fontsize=7.5)
        ax1.set_title(
            f"{var}\n(IV={iv_val:.3f})", fontsize=8.5, fontweight="bold", pad=4
        )

        # ── Count annotation on each bar ────────────────────────────
        for bar, cnt in zip(bars, counts):
            ht = bar.get_height()
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                ht + (0.01 if ht >= 0 else -0.03),
                f"n={int(cnt)}",
                ha="center",
                va="bottom",
                fontsize=6.5,
                color="#333",
            )

        # ── Line: event rate (dual axis) ────────────────────────────
        ax2 = ax1.twinx()
        ax2.plot(
            x,
            bad_rt,
            color="#e67e22",
            marker="o",
            markersize=4,
            linewidth=1.5,
            zorder=3,
            label="Bad rate",
        )
        ax2.set_ylabel("Bad rate", fontsize=8, color="#e67e22")
        ax2.tick_params(axis="y", labelsize=7, colors="#e67e22")
        ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

        ax1.grid(axis="y", linestyle=":", linewidth=0.5, alpha=0.6)

    # ── Hide unused subplots ─────────────────────────────────────────
    for j in range(idx + 1, len(axes)):
        axes[j].set_visible(False)

    title = f"{seg_name} — WOE bin plots" if seg_name else "WOE bin plots"
    fig.suptitle(title, fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()

    if save_dir:
        path = f"{save_dir}/{seg_name}_woebin_plots.png"
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print(f"Saved: {path}")

    plt.show()

In [ ]:
def assign_segment(df: pd.DataFrame) -> pd.Series:
    conditions = [
        df["loan_tenure"] <= 90,
        df["loan_tenure"] >= 91,
    ]
    labels = ["new_loans", "old_loans"]
    return pd.Series(
        np.select(conditions, labels, default="all_loans"),
        index=df.index,

        dtype="object",    )

In [ ]:
df["segment"] = assign_segment(df)
df["segment"].value_counts()

In [ ]:
df_train['bad']

## All loans

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import pickle
# import matplotlib.pyplot as plt

# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import roc_auc_score
# from sklearn.inspection import permutation_importance
# from statsmodels.stats.outliers_influence import variance_inflation_factor

# out_dir = OUT_DIR
# group = {
#     "base": "base_",
#     "cdr": "cdr_",
#     "mobile": "mobile_",
#     "penalty": "penalty_",
#     "bills": "bills_",
# }

# WINDOWS = ["372_188", "187_98", "97_38", "37_8", "30_60", "30_90", "60_90"]

# MAX_PER_WINDOW = 2
# MAX_PER_GROUP = 10


# # ─────────────────────────────────────────────────────────────
# # MAIN LOOP
# # ─────────────────────────────────────────────────────────────
# for seg_name, df_seg in segments.items():

#     if seg_name != "all_loans":
#         continue

#     # ── Setup directory ───────────────────────────────────────
#     seg_dir = f"{out_dir}/{seg_name}"
#     os.makedirs(seg_dir, exist_ok=True)

#     print(f"\n========== {seg_name} ==========\n")

#     df_seg = df_seg.drop(columns=["data_type", "base_date"]).copy()
#     df_seg["bad"] = df_seg["bad"].astype(int)

#     # ── Data prep ─────────────────────────────────────────────
#     for c in df_seg.columns:
#         if c in ["data_type", "base_date", "base_late_paid_1_7", "base_late_paid_8_14", "base_late_paid_15_30", "base_late_paid_31_60", "base_late_paid_61_90"]:
#             df_seg = df_seg.drop(columns=[c], inplace=False)
#     df_seg["bad"] = df_seg["bad"].astype(int)

#     #df_seg.drop(columns=["base_rep_cnt_187_98"], inplace=True)

#     dt = sc.var_filter(df_seg, y="bad")
#     dt.to_csv(f'{seg_dir}/{seg_name}_var_filter.csv', index=False)
#     dt = pd.read_csv(f'{seg_dir}/{seg_name}_var_filter.csv')

#     train, test = sc.split_df(dt, y="bad", ratio=0.8, seed=42).values()

#     raw_train = train.copy()
#     raw_test = test.copy()

#     bins  = sc.woebin(train, y="bad")


#     # ── IV filter ─────────────────────────────────────────────
#     iv_df = pd.DataFrame(
#         {var: bins[var]["total_iv"].iloc[0] for var in bins}, index=["IV"]
#     ).T
#     print(iv_df.head())

#     selected_vars = (
#         iv_df[(iv_df["IV"] >= 0.02) & (iv_df["IV"] <= 0.50)]
#         .sort_values("IV", ascending=False)
#         .index.tolist()
#     )[:80]

#     # ── Group deduplication ───────────────────────────────────
#     # Rule 1: if a SOURCE GROUP prefix (e.g. 'base_', 'cdr_') has >3 features,
#     #         keep only top-1 IV per WINDOW SUFFIX (e.g. '372_188', '37_8')
#     # Rule 2: if a WINDOW SUFFIX across ALL groups has >3 features,
#     #         keep only top-1 IV within that window bucket

#     WINDOWS = ["372_188", "187_98", "97_38", "37_8", "30_60", "60_90", "30_90"]

#     iv_selected = iv_df.loc[selected_vars, "IV"]  # Series, sorted desc

#     # def get_prefix(col):
#     #     for prefix in group.values():
#     #         if col.startswith(prefix):
#     #             return prefix
#     #     return "other"

#     # def get_window(col):
#     #     for w in WINDOWS:
#     #         if w in col:
#     #             return w
#     #     return None

#     # seen_window_count = {}  # window  → count of features kept so far
#     # seen_group_count = {}  # prefix  → count of features kept so far
#     # final_selected = []

#     # for col in iv_selected.index:
#     #     w = get_window(col)
#     #     p = get_prefix(col)

#     #     window_ok = (w is None) or (seen_window_count.get(w, 0) < MAX_PER_WINDOW)
#     #     group_ok = seen_group_count.get(p, 0) < MAX_PER_GROUP

#     #     if window_ok and group_ok:
#     #         final_selected.append(col)
#     #         if w:
#     #             seen_window_count[w] = seen_window_count.get(w, 0) + 1
#     #         seen_group_count[p] = seen_group_count.get(p, 0) + 1

#     # selected_vars = final_selected
#     # print(f"\nAfter group dedup: {len(selected_vars)} features")
#     # print(pd.Series({c: round(iv_selected[c], 4) for c in selected_vars}))

#     # # debug — verify limits
#     # from collections import Counter

#     # w_counts = Counter(get_window(c) for c in selected_vars if get_window(c))
#     # g_counts = Counter(get_prefix(c) for c in selected_vars)
#     # print(f"\nWindow counts (max allowed {MAX_PER_WINDOW}):", dict(w_counts))
#     # print(f"Group counts  (max allowed {MAX_PER_GROUP}):", dict(g_counts))

#     # ── Continue with filtered vars ───────────────────────────
#     bins = {k: bins[k] for k in selected_vars if k in bins}
#     train = train[selected_vars + ["bad"]]
#     test = test[selected_vars + ["bad"]]

#     # ── WOE transform ─────────────────────────────────────────
#     train_woe = sc.woebin_ply(train, bins)
#     test_woe = sc.woebin_ply(test, bins)

#     y_train = train_woe["bad"]
#     X_train = train_woe.drop(columns=["bad"])
#     y_test = test_woe["bad"]
#     X_test = test_woe.drop(columns=["bad"])

#     FLOOR     = 12
#     FINAL_CAP = 15

#     # ── Correlation filter — floor-aware ──────────────────────
#     corr  = X_train.corr().abs()
#     upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

#     # sort candidates by how many high-corr pairs they appear in (drop busiest first)
#     corr_counts = (upper > 0.75).sum().sort_values(ascending=False)
#     drop_corr = []
#     for col in corr_counts.index:
#         if X_train.shape[1] - len(drop_corr) <= FLOOR:
#             break
#         remaining = [c for c in upper.columns if c not in drop_corr]
#         if col in remaining and any(upper.loc[remaining, col] > 0.75):
#             drop_corr.append(col)

#     X_train = X_train.drop(columns=drop_corr)
#     X_test  = X_test[X_train.columns]
#     print(f"After corr : {X_train.shape[1]}")

#     # ── VIF filter — floor-aware ───────────────────────────────
#     while X_train.shape[1] > FLOOR:
#         vif = pd.Series(
#             [variance_inflation_factor(X_train.values, i)
#             for i in range(X_train.shape[1])],
#             index=X_train.columns
#         )
#         if vif.max() < 10:
#             break
#         X_train = X_train.drop(columns=[vif.idxmax()])
#         X_test  = X_test[X_train.columns]
#     print(f"After VIF  : {X_train.shape[1]}")

#     # ── L1 selection — floor-aware ─────────────────────────────
#     scaler  = StandardScaler()
#     X_tr_sc = scaler.fit_transform(X_train)
#     lr_l1   = LogisticRegression(penalty="l2", C=0.5, solver="saga", max_iter=500)
#     lr_l1.fit(X_tr_sc, y_train)

#     coef_abs  = pd.Series(np.abs(lr_l1.coef_[0]), index=X_train.columns).sort_values(ascending=False)
#     l1_kept   = X_train.columns[lr_l1.coef_[0] != 0].tolist()

#     if len(l1_kept) < FLOOR:
#         # restore highest |coef| features until floor is met
#         l1_kept = coef_abs.head(FLOOR).index.tolist()
#         print(f"⚠ L1 floor guard: kept top-{FLOOR} by |coef|")

#     X_train = X_train[l1_kept]
#     X_test  = X_test[l1_kept]
#     print(f"After L1   : {X_train.shape[1]}")

#     # ── Permutation importance — floor-aware ───────────────────
#     lr_probe = LogisticRegression(penalty="l2", C=0.5, solver="saga", max_iter=500)
#     lr_probe.fit(X_train, y_train)

#     perm = permutation_importance(
#         lr_probe, X_train, y_train,
#         scoring="roc_auc", n_repeats=10, random_state=42
#     )
#     perm_df = pd.DataFrame({
#         "feature":    X_train.columns,
#         "importance": perm.importances_mean
#     }).sort_values("importance", ascending=False)

#     keep_perm = perm_df[perm_df["importance"] > 0]["feature"].tolist()
#     if len(keep_perm) < FLOOR:
#         keep_perm = perm_df["feature"].tolist()[:FLOOR]
#         print(f"⚠ Perm floor guard: kept top-{FLOOR} by importance")

#     X_train = X_train[keep_perm]
#     X_test  = X_test[keep_perm]
#     print(f"After perm : {X_train.shape[1]}")

#     # ── Cap at FINAL_CAP ───────────────────────────────────────
#     if X_train.shape[1] > FINAL_CAP:
#         keep_cap = perm_df[perm_df["feature"].isin(X_train.columns)]["feature"].tolist()[:FINAL_CAP]
#         X_train  = X_train[keep_cap]
#         X_test   = X_test[keep_cap]
#     print(f"After cap  : {X_train.shape[1]}")

#     # ── Negative coefficient removal — floor-aware ─────────────
#     lr_check = LogisticRegression(penalty="l2", C=0.5, solver="saga", max_iter=500)
#     lr_check.fit(X_train, y_train)

#     coef_s       = pd.Series(lr_check.coef_[0], index=X_train.columns).sort_values()
#     neg_features = coef_s[coef_s < 0].index.tolist()

#     # only drop negatives if we stay at or above floor
#     n_droppable  = max(0, X_train.shape[1] - FLOOR)
#     neg_to_drop  = neg_features[:n_droppable]   # most negative first

#     if len(neg_features) > n_droppable:
#         kept_neg = neg_features[n_droppable:]
#         print(f"⚠ Neg-coef floor guard: kept {len(kept_neg)} negative features to hold floor={FLOOR}")
#         print(f"  Kept (least negative): {kept_neg}")

#     X_train = X_train.drop(columns=neg_to_drop)
#     X_test  = X_test[X_train.columns]
#     print(f"After neg  : {X_train.shape[1]}  ← target {FLOOR}–{FINAL_CAP}")

#     # ── Final model ───────────────────────────────────────────
#     lr_final =LogisticRegression(penalty="l2", C=0.5, solver="saga", max_iter=500)
#     lr_final.fit(X_train, y_train)

#     final_features = [c.replace("_woe", "") for c in X_train.columns]
#     final_bins = {k: bins[k] for k in final_features if k in bins}

#     print(f"\nFINAL feature count: {len(final_features)}")

#     # ── Evaluation ────────────────────────────────────────────
#     train_pred = lr_final.predict_proba(X_train)[:, 1]
#     test_pred = lr_final.predict_proba(X_test)[:, 1]

#     train_auc = roc_auc_score(y_train, train_pred)
#     test_auc = roc_auc_score(y_test, test_pred)

#     sc.perf_eva(
#         y_train, lr_final.predict_proba(X_train)[:, 1], title=f"{seg_name}_train"
#     )
#     plt.savefig(
#         f"{seg_dir}/{seg_name}_perf_train_plot.png", dpi=300, bbox_inches="tight"
#     )

#     sc.perf_eva(y_test, lr_final.predict_proba(X_test)[:, 1], title=f"{seg_name}_test")
#     plt.savefig(
#         f"{seg_dir}/{seg_name}_perf_test_plot.png", dpi=300, bbox_inches="tight"
#     )

#     print("Train AUC:", round(train_auc, 4))
#     print("Test  AUC:", round(test_auc, 4))

#     # ── Scorecard ─────────────────────────────────────────────
#     card = sc.scorecard(final_bins, lr_final, X_train.columns)
#     card_df = pd.concat(card).reset_index(drop=True)
#     card_df.to_csv(f"{seg_dir}/{seg_name}_scorecard_full.csv", index=False)

#     card_save = (
#         pd.concat(card)
#         .groupby(["variable"])["points"]
#         .agg(["min", "max", "count"])
#         .reset_index()
#     )
#     card_save["score_range"] = np.abs(card_save["min"]) + np.abs(card_save["max"])
#     card_save = card_save.sort_values("score_range", ascending=False)
#     card_save.to_csv(f"{seg_dir}/{seg_name}_scorecard.csv", index=False)

#     surviving_raw = final_features + ["bad"]
#     raw_train_final = raw_train[surviving_raw]
#     raw_test_final = raw_test[surviving_raw]

#     train_score = sc.scorecard_ply(raw_train_final, card, print_step=0)
#     test_score = sc.scorecard_ply(raw_test_final, card, print_step=0)

#     sc.woebin_plot(final_bins)

#     plot_woe_bins(final_bins, seg_name=seg_name, save_dir=seg_dir)
#     # ── Save outputs ──────────────────────────────────────────
#     pd.Series(final_features).to_csv(
#         f"{seg_dir}/{seg_name}_final_features.csv", index=False
#     )
#     iv_df.sort_values("IV", ascending=False).to_csv(
#         f"{seg_dir}/{seg_name}_iv.csv"
#     )
#     X_train.corr().to_csv(f"{seg_dir}/{seg_name}_corr_matrix.csv")
#     X_train.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.to_csv(
#         f"{seg_dir}/{seg_name}_feature_distribution.csv"
#     )
#     score_distribution_plot(train_score, test_score, y_train, y_test, seg_name, seg_dir)
#     cutoff_analysis(test_score, y_test, seg_name, seg_dir)

#     audit = {
#         "segment": seg_name,
#         "n_features_final": len(final_features),
#         "features": list(final_features),
#         "n_train": len(train),
#         "n_test": len(test),
#         "target_rate_train": float(y_train.mean()),
#         "target_rate_test": float(y_test.mean()),
#     }

#     pd.Series(audit).to_json(f"{seg_dir}/{seg_name}_audit.json", indent=2)

#     pickle.dump(lr_final, open(f"{seg_dir}/{seg_name}_model.pkl", "wb"))
#     pickle.dump(card, open(f"{seg_dir}/{seg_name}_card.pkl", "wb"))

#     pickle.dump(final_bins, open(f"{seg_dir}/{seg_name}_bins.pkl", "wb"))


## All Loans FIxed

In [ ]:
import numpy as np
import pandas as pd
import pickle
import scorecardpy as sc
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import KBinsDiscretizer

seg_name = "all_loans"
out_dir = OUT_DIR
seg_dir = f"{out_dir}/{seg_name}"
os.makedirs(seg_dir, exist_ok=True)

print(f"\n========== {seg_name} ==========\n")

# ── Data prep ─────────────────────────────────────────────

df_all = df_train.copy()

# ── 1. LOCKED CONFIG ──────────────────────────────────────────────────
ALL_LOCKED_FEATURES = [
   "base_max_dpd_97_38",
   "base_stddev_days_late_all",
   "base_ontime_cnt_97_38",
   "invoice_ontime_cnt_ratio_stability",
   "base_avg_dpd_187_98",
   "base_rep_cnt_187_98",
   "base_avg_dpd_chg_37_8_vs_97_38",
   "base_max_dpd_acceleration",
   "app_trx_recency_l90d",
   "cdr_consistency_index",
   "base_recent_payment_concentration",
   "base_unpaid_rate_chg_187_98_vs_372_188",
  # "base_max_dpd_chg_97_38_vs_187_98",
   #"base_unpaid_cnt_372_188"
]
print(ALL_LOCKED_FEATURES)

# Optional manual bin edges per feature, e.g.
#   "base_max_dpd_97_38": [-np.inf, 10, 30, np.inf],
# Leave empty to let tree-based binning choose the breaks.
ALL_MANUAL_BREAKS = {}

# ── 2. DATA PREP ──────────────────────────────────────────────────────
df_all = df_all[[*ALL_LOCKED_FEATURES, "bad"]].copy()


print(df_all.columns)

df_all.fillna(0, inplace=True)
df_all["bad"] = df_all["bad"].astype(int)
dt_all = sc.var_filter(df_all, y="bad")
train_all, test_all = sc.split_df(dt_all, y="bad", ratio=0.8, seed=42).values()

raw_train = train_all.copy()
raw_test = test_all.copy()

# ── 3. BINNING ────────────────────────────────────────────────────────
bins = sc.woebin(
    train_all, y="bad", method="tree", min_perc_bin=0.05, breaks_list=ALL_MANUAL_BREAKS
)

locked_bins = {k: bins[k] for k in ALL_LOCKED_FEATURES if k in bins}
missing = [k for k in ALL_LOCKED_FEATURES if k not in bins]
if missing:
    raise ValueError(f"These locked features are missing from bins: {missing}")

# ── 4. WOE TRANSFORM ─────────────────────────────────────────────────
locked_woe_cols = [f + "_woe" for f in ALL_LOCKED_FEATURES]

iv_df = pd.DataFrame(
    {var: bins[var]["total_iv"].iloc[0] for var in bins}, index=["IV"]
).T

train_woe = sc.woebin_ply(train_all[ALL_LOCKED_FEATURES + ["bad"]], locked_bins)
test_woe = sc.woebin_ply(test_all[ALL_LOCKED_FEATURES + ["bad"]], locked_bins)

print(train_woe.isnull().sum())

y_train = train_woe["bad"]
X_train = train_woe[locked_woe_cols]
y_test = test_woe["bad"]
X_test = test_woe[locked_woe_cols]

print("Train WOE shape:", X_train.shape)
print("final features:", locked_woe_cols)


# ── 5. TRAIN ──────────────────────────────────────────────────────────
lr_final = LogisticRegression(penalty="l2", C=0.5, solver="saga", max_iter=500)
lr_final.fit(X_train, y_train)

# ── 6. EVALUATE ───────────────────────────────────────────────────────
train_auc = roc_auc_score(y_train, lr_final.predict_proba(X_train)[:, 1])
test_auc = roc_auc_score(y_test, lr_final.predict_proba(X_test)[:, 1])

print(f"Train AUC : {train_auc:.4f}")
print(f"Test  AUC : {test_auc:.4f}")

sc.perf_eva(y_train, lr_final.predict_proba(X_train)[:, 1], title=f"{seg_name}_train")
sc.perf_eva(y_test, lr_final.predict_proba(X_test)[:, 1], title=f"{seg_name}_test")


# ── 7. SCORECARD ──────────────────────────────────────────────────────
card = sc.scorecard(locked_bins, lr_final, locked_woe_cols)

raw_train_final = raw_train[ALL_LOCKED_FEATURES + ["bad"]]
raw_test_final = raw_test[ALL_LOCKED_FEATURES + ["bad"]]

train_score = sc.scorecard_ply(raw_train_final, card, print_step=0)
test_score = sc.scorecard_ply(raw_test_final, card, print_step=0)

plot_woe_bins(locked_bins, seg_name=seg_name, save_dir=seg_dir)#
#save_woebin_plots(locked_bins, seg_name=seg_name, seg_dir=seg_dir)
sc.woebin_plot(locked_bins)

# ── 7.5 REPORT JSON ───────────────────────────────────────────────────
import json
from datetime import datetime

train_pred = lr_final.predict_proba(X_train)[:, 1]
test_pred = lr_final.predict_proba(X_test)[:, 1]


def ks_stat(y, p):
    df = pd.DataFrame({"y": y.values, "p": p}).sort_values("p", ascending=False)
    df["cum_bad"] = df["y"].cumsum() / df["y"].sum()
    df["cum_good"] = (1 - df["y"]).cumsum() / (1 - df["y"]).sum()
    return round(float((df["cum_bad"] - df["cum_good"]).abs().max()), 4)


# ── Consistent binning across train and test ─────────────────────────
kbd = KBinsDiscretizer(n_bins=10, encode="ordinal", strategy="kmeans")


def band_summary_csv(score_df, y):
    y_vals = y.values if hasattr(y, "values") else np.array(y)
    tmp = score_df[["score", "bins"]].copy()
    tmp["bad"] = y_vals

    bin_stats = (
        tmp.groupby("bins")
        .agg(
            min_score=("score", "min"),
            max_score=("score", "max"),
            total_count=("bad", "size"),
            good_count=("bad", lambda x: (x == 0).sum()),
            bad_count=("bad", lambda x: (x == 1).sum()),
        )
        .reset_index()
    )
    bin_stats["bad_rate"]      = (bin_stats["bad_count"] / bin_stats["total_count"]).round(4)
    bin_stats["good_bad_odds"] = (bin_stats["good_count"] / bin_stats["bad_count"].replace(0, np.nan)).round(2)
    bin_stats["dist_rate"]     = (bin_stats["total_count"] / len(tmp)).round(4)

    return bin_stats.sort_values("bins").reset_index(drop=True)

def band_summary(score_df, y):
    s = score_df["score"].values
    s_min, s_max = int(s.min()), int(s.max())
    step = max(10, (s_max - s_min) // 15)
    edges = list(range(s_min, s_max + step, step))
    out = []
    tmp = pd.DataFrame({"score": s, "bad": y.values})
    for i in range(len(edges) - 1):
        chunk = tmp[(tmp["score"] >= edges[i]) & (tmp["score"] < edges[i + 1])]
        out.append(
            {
                "band": f"{edges[i]}–{edges[i+1]}",
                "count": int(len(chunk)),
                "bad": int(chunk["bad"].sum()),
                "bad_rate": round(float(chunk["bad"].mean()), 4) if len(chunk) else 0,
            }
        )
    return out


# feature log
feature_log = []
for var in locked_bins:
    df_bin = locked_bins[var].copy()
    woe_col = var + "_woe"
    coef_idx = (
        list(X_train.columns).index(woe_col) if woe_col in X_train.columns else None
    )
    coef_val = float(lr_final.coef_[0][coef_idx]) if coef_idx is not None else None

    bin_rows = []
    for _, row in df_bin.iterrows():
        bin_rows.append(
            {
                "bin": str(row.get("bin", "")),
                "count": int(row.get("count", 0)),
                "count_pct": round(float(row.get("count_distr", 0)) * 100, 2),
                "good": int(row.get("good", 0)),
                "bad": int(row.get("bad", 0)),
                "bad_rate": round(float(row.get("badprob", 0)), 4),
                "woe": round(float(row.get("woe", 0)), 4),
                "iv": round(float(row.get("iv", 0)), 4),
            }
        )

    bp = [b["bad_rate"] for b in bin_rows if b["bin"] != "missing"]
    is_up = all(bp[i] <= bp[i + 1] for i in range(len(bp) - 1))
    is_down = all(bp[i] >= bp[i + 1] for i in range(len(bp) - 1))
    mono = (
        "↑ ascending" if is_up else ("↓ descending" if is_down else "⚠ non-monotonic")
    )

    feature_log.append(
        {
            "feature": var,
            "total_iv": round(float(df_bin["total_iv"].iloc[0]), 4),
            "coef": round(coef_val, 4) if coef_val is not None else None,
            "n_bins": len(bin_rows),
            "monotonic": mono,
            "bins": bin_rows,
        }
    )
feature_log.sort(key=lambda x: x["total_iv"], reverse=True)

# scorecard log
card_log = []
for var, df_card in card.items():
    if var == "basepoints":
        continue
    pts = df_card["points"].dropna()
    card_log.append(
        {
            "variable": var,
            "min_pts": int(pts.min()),
            "max_pts": int(pts.max()),
            "spread": int(pts.max() - pts.min()),
            "bins": df_card[["bin", "points"]].fillna("").to_dict("records"),
        }
    )
card_log.sort(key=lambda x: x["spread"], reverse=True)

# perf log
tr_auc = round(roc_auc_score(y_train, train_pred), 4)
te_auc = round(roc_auc_score(y_test, test_pred), 4)
perf_log = {
    "train": {
        "n": int(len(y_train)),
        "bad_rate": round(float(y_train.mean()), 4),
        "auc": tr_auc,
        "gini": round(2 * tr_auc - 1, 4),
        "ks": ks_stat(y_train, train_pred),
    },
    "test": {
        "n": int(len(y_test)),
        "bad_rate": round(float(y_test.mean()), 4),
        "auc": te_auc,
        "gini": round(2 * te_auc - 1, 4),
        "ks": ks_stat(y_test, test_pred),
    },
    "overfit_gap": round(tr_auc - te_auc, 4),
}

train_score["bins"] = 10 - kbd.fit_transform(train_score[["score"]]).astype(int).flatten()
test_score["bins"]  = 10 - kbd.transform(test_score[["score"]]).astype(int).flatten()

with open(f"{seg_dir}/bin_edges.json", "w") as f:
    json.dump([edges.tolist() for edges in kbd.bin_edges_], f)


train_bands = band_summary_csv(train_score, y_train)
test_bands  = band_summary_csv(test_score,  y_test)
# ── Save CSVs ─────────────────────────────────────────────────────────
train_bands.to_csv(f"{seg_dir}/{seg_name}_train_score_bands.csv", index=False)
test_bands.to_csv(f"{seg_dir}/{seg_name}_test_score_bands.csv",  index=False)
print(f"Saved score band CSVs → {seg_dir}")
# score log
score_log = {
    "train": band_summary_csv(train_score, y_train),
    "test": band_summary_csv(test_score, y_test),
    "train_score_stats": {
        k: round(float(v), 2) for k, v in train_score["score"].describe().items()
    },
    "test_score_stats": {
        k: round(float(v), 2) for k, v in test_score["score"].describe().items()
    },
}

# model log
model_log = {
    "model_type": "LogisticRegression",
    "penalty": lr_final.penalty,
    "C": lr_final.C,
    "solver": lr_final.solver,
    "max_iter": lr_final.max_iter,
    "n_features": len(locked_woe_cols),
    "intercept": round(float(lr_final.intercept_[0]), 6),
    "coefficients": {
        col: round(float(c), 6) for col, c in zip(locked_woe_cols, lr_final.coef_[0])
    },
    "scorecard_params": {"points0": 600, "odds0": "1/19", "pdo": 50},
    "feature_selection": {"method": "manual", "locked_features": ALL_LOCKED_FEATURES},
}

logs = {
    "meta": {
        "segment": seg_name,
        "generated": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "n_features": len(ALL_LOCKED_FEATURES),
    },
    "features": feature_log,
    "model": model_log,
    "performance": perf_log,
    "scores": score_log,
    "scorecard": card_log,
}

json_path = f"{seg_dir}/{seg_name}_report_data.json"
with open(json_path, "w") as f:
    json.dump(logs, f, indent=2, default=str)
print(f"Report JSON → {json_path}")

# ── 8. SAVE ───────────────────────────────────────────────────────────
pd.Series(ALL_LOCKED_FEATURES).to_csv(
    f"{seg_dir}/{seg_name}_final_features.csv", index=False
)

card_df = pd.concat({k: v for k, v in card.items() if k != "basepoints"}).reset_index(drop=True)
card_df.to_csv(f"{seg_dir}/{seg_name}_scorecard_full.csv", index=False)

card_agg = (
    card_df.groupby("variable")["points"].agg(["min", "max", "count"]).reset_index()
)
card_agg["score_range"] = np.abs(card_agg["min"]) + np.abs(card_agg["max"])
card_agg = card_agg.sort_values("score_range", ascending=False)
card_agg.to_csv(f"{seg_dir}/{seg_name}_scorecard.csv", index=False)

iv_df.rename_axis("feature").reset_index().to_csv(
    f"{seg_dir}/{seg_name}_iv.csv", index=False
)
X_train.corr().to_csv(f"{seg_dir}/{seg_name}_corr_matrix.csv")
X_train.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.to_csv(
    f"{seg_dir}/{seg_name}_feature_distribution.csv"
)
score_distribution_plot(train_score, test_score, y_train, y_test, seg_name, seg_dir)
cutoff_analysis(test_score, y_test, seg_name, seg_dir)
audit = {
    "segment": seg_name,
    "n_features_final": len(ALL_LOCKED_FEATURES),
    "features": ALL_LOCKED_FEATURES,
    "n_train": len(train_all),
    "n_test": len(test_all),
    "target_rate_train": float(y_train.mean()),
    "target_rate_test": float(y_test.mean()),
    "train_auc": round(train_auc, 4),
    "test_auc": round(test_auc, 4),
}
pd.Series(audit).to_json(f"{seg_dir}/{seg_name}_audit.json", indent=2)

pickle.dump(lr_final, open(f"{seg_dir}/{seg_name}_model.pkl", "wb"))
pickle.dump(card, open(f"{seg_dir}/{seg_name}_card.pkl", "wb"))
pickle.dump(locked_bins, open(f"{seg_dir}/{seg_name}_bins.pkl", "wb"))
pickle.dump(kbd, open(f"{seg_dir}/{seg_name}_kbd.pkl", "wb"))  # Also save the fitted kbd object


print("Done. Artifacts saved to", seg_dir)

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import scorecardpy as sc
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import roc_auc_score, roc_curve

n_bins = 10
base_seg = "all_loans"
out_dir = OUT_DIR
artifacts_dir = f"{out_dir}/{base_seg}"

def ks_stat(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))

def score_unseen(df_raw, label, seg_name, art, out_dir):
    feats = art["features"]
    bins = art["bins"]
    model = art["model"]
    card = art["card"]
    kbd = art["kbd"]  # Load the fitted kbd

    df_seg = df_raw.copy()
    if df_seg.empty:
        print(f"  [{label}] {seg_name}: no borrowers — skipping")
        return

    df_seg["bad"] = df_seg["bad"].astype(int)

    df_feats = df_seg[feats].copy()
    df_woe = sc.woebin_ply(df_feats, bins)
    model_cols = [f + "_woe" for f in feats]
    df_woe = df_woe[model_cols]

    unseen_score = model.predict_proba(df_woe)[:, 1]
    auc_val = roc_auc_score(df_seg["bad"], unseen_score)
    ks_val = ks_stat(df_seg["bad"], unseen_score)
    gini_val = 2 * auc_val - 1

    print(f"  [{label}] {seg_name}  AUC={auc_val:.4f}  KS={ks_val:.4f}")

    df_seg_model = df_feats.copy()
    df_seg_model["bad"] = df_seg["bad"].values

    df_seg["score"] = sc.scorecard_ply(df_seg_model, card, print_step=0)
    # Use the saved kbd to transform (not fit) - ensures consistency
    df_seg["bins"] = 10 - kbd.transform(df_seg[["score"]]).astype(int).flatten()

    bin_stats = (
        df_seg.groupby("bins")
        .agg(
            min_score=("score", "min"),
            max_score=("score", "max"),
            total_count=("bad", "size"),
            good_count=("bad", lambda x: (x == 0).sum()),
            bad_count=("bad", lambda x: (x == 1).sum()),
        )
        .reset_index()
    )
    bin_stats["bad_rate"] = bin_stats["bad_count"] / bin_stats["total_count"]
    bin_stats["good_bad_odds"] = bin_stats["good_count"] / bin_stats["bad_count"]

    seg_dir = f"{out_dir}/{seg_name}"
    os.makedirs(seg_dir, exist_ok=True)
    bin_stats.to_csv(f"{seg_dir}/{seg_name}_{label}_bins.csv", index=False)

    unseen_entry = {
        "label": label,
        "n": int(len(df_seg)),
        "bad_rate": round(float(df_seg["bad"].mean()), 4),
        "auc": round(auc_val, 4),
        "ks": round(ks_val, 4),
        "gini": round(gini_val, 4),
        "bands": [
            {
                "band": int(row["bins"]),
                "count": int(row["total_count"]),
                "bad": int(row["bad_count"]),
                "bad_rate": round(float(row["bad_rate"]), 4),
            }
            for _, row in bin_stats.sort_values("bins").iterrows()
        ],
    }

    report_path = f"{seg_dir}/{seg_name}_report_data.json"
    if os.path.exists(report_path):
        with open(report_path) as f:
            report = json.load(f)
        report.setdefault("unseen", [])
        report["unseen"] = [u for u in report["unseen"] if u["label"] != label]
        report["unseen"].append(unseen_entry)
        with open(report_path, "w") as f:
            json.dump(report, f, indent=2)
    else:
        print(f"  [{label}] WARNING: {report_path} not found — JSON not updated")

artifacts = {
    "model": pickle.load(open(f"{artifacts_dir}/{base_seg}_model.pkl", "rb")),
    "card": pickle.load(open(f"{artifacts_dir}/{base_seg}_card.pkl", "rb")),
    "bins": pickle.load(open(f"{artifacts_dir}/{base_seg}_bins.pkl", "rb")),
    "kbd": pickle.load(open(f"{artifacts_dir}/{base_seg}_kbd.pkl", "rb")),  # Load fitted kbd
    "features": (
        pd.read_csv(f"{artifacts_dir}/{base_seg}_final_features.csv")["0"]
        .str.replace("_woe", "")
        .tolist()
    ),
}

for name, df_raw in [("df_test4", df_test4), ("df_test5", df_test5)]:
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    score_unseen(df_raw.copy(), name, base_seg, artifacts, out_dir)

## Score all

In [ ]:
def score_saved_model(df, label_col=None, prefix="unseen"):
    df_keep = df[['base_date', 'userid', 'borrowerid', 'loanid']]
    df_proc = df[ALL_LOCKED_FEATURES].copy()
    df_proc = df_proc.fillna(0)

    # Apply the same WOE transformation used for training
    df_woe = sc.woebin_ply(df_proc, locked_bins)
    X_score = df_woe[locked_woe_cols]

    # Score with the trained model and the saved scorecard
    score_df = sc.scorecard_ply(df_proc, card, print_step=0)
    score_df["bins"] = 10 - kbd.transform(score_df[["score"]]).astype(int).flatten()


    score_df["probability"] = lr_final.predict_proba(X_score)[:, 1]

    score_df = pd.concat([df_keep, score_df], axis=1)
    out_path = os.path.join(seg_dir, f"{seg_name}_{prefix}_scores.csv")
    score_df.to_csv(out_path, index=False)
    print("saved to ", out_path)

    if label_col is not None:
        labels = df[label_col].astype(int)
        score_df[label_col] = labels.values
        # bands = band_summary_csv(score_df, labels)
        # bands.to_csv(os.path.join(seg_dir, f"{seg_name}_{prefix}_score_bands.csv"), index=False)

        perf = {
            "n": int(len(labels)),
            "bad_rate": round(float(labels.mean()), 4),
            "auc": round(float(roc_auc_score(labels, score_df["probability"])), 4),
            "gini": round(2 * float(roc_auc_score(labels, score_df["probability"])) - 1, 4),
            "ks": ks_stat(labels, score_df["probability"]),
        }
        with open(os.path.join(seg_dir, f"{seg_name}_{prefix}_perf.json"), "w") as f:
            json.dump(perf, f, indent=2)

    return score_df

# Example usage:
score_saved_model(df_target, label_col="bad", prefix="all")
# score_saved_model(raw_test_final, label_col="bad", prefix="test_seen")
# score_saved_model(df_unseen, label_col=None, prefix="unseen")


## Score additional pool (recent period)

In [ ]:
df = pd.read_csv(f"{OUT_DIR}/all_loans/all_loans_all_scores.csv")

In [ ]:
df

In [ ]:
# Pool of loans to score; raw feature names are mapped via FEATURE_RENAME_MAP below
pool = load_table("collection_pool")
pool = pool[pd.to_datetime(pool["p_date"]) >= "2023-12-01"]

In [ ]:
pool.drop_duplicates(subset=['loanid', 'p_date'], inplace=True)
pool.dropna(subset=['loanid', 'p_date'])

In [ ]:
print(pool.shape)
print(pool.head())

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import scorecardpy as sc
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import roc_auc_score, roc_curve

n_bins = 10
base_seg = "all_loans"
out_dir = OUT_DIR
artifacts_dir = f"{out_dir}/{base_seg}"

META_COLS = ["p_date", "userid", "borrowerid", "loanid"]

FEATURE_RENAME_MAP = {
    "max_dpd_97_38": "base_max_dpd_97_38",
    "stddev_days_late_all": "base_stddev_days_late_all",
    "ontime_cnt_97_38": "base_ontime_cnt_97_38",
    "avg_dpd_187_98": "base_avg_dpd_187_98",
    "rep_cnt_187_98": "base_rep_cnt_187_98",
    "avg_dpd_chg_37_8_vs_97_38": "base_avg_dpd_chg_37_8_vs_97_38",
    "max_dpd_acceleration": "base_max_dpd_acceleration",
    "trx_recency_l90d": "app_trx_recency_l90d",
    "consistency_index": "cdr_consistency_index",
    "invoice_ontime_cnt_ratio_stability": "invoice_ontime_cnt_ratio_stability",
    "recent_payment_concentration": "base_recent_payment_concentration",
    "unpaid_rate_chg_187_98_vs_372_188": "base_unpaid_rate_chg_187_98_vs_372_188",
}

def prepare_features(df: pd.DataFrame, feature_list: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    df_feat = df.copy()
    df_feat.rename(columns=FEATURE_RENAME_MAP, inplace=True)

    missing_meta = [c for c in META_COLS if c not in df_feat.columns]
    if missing_meta:
        print("Missing meta columns: %s — will be skipped", missing_meta)

    missing_feats = [f for f in feature_list if f not in df_feat.columns]
    if missing_feats:
        raise ValueError(
            f"Feature mismatch — these features are in the model but not in input data:\n"
            f"  {missing_feats}"
        )

    df_meta = df_feat[[c for c in META_COLS if c in df_feat.columns]].copy()
    df_features = df_feat[feature_list].copy()
    for c in df_features.columns:
        if c == "app_trx_recency_l90d":
            df_features[c] = df_features[c].fillna(91)
        else:
            df_features[c] = df_features[c].fillna(0)

    df_features.replace("None", 0, inplace=True)
    df_features.replace([np.inf, -np.inf], 0, inplace=True)

    print(
        "Features prepared: %d rows x %d features | NaN filled with 0",
        len(df_features), len(feature_list),
    )
    return df_meta, df_features


def ks_stat(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))

def score_pool(df_raw, art, out_dir):
    feature_list = art["features"]
    bins = art["bins"]
    model = art["model"]
    card = art["card"]
    kbd = art["kbd"]  # Load the fitted kbd


    df_meta, df_feat = prepare_features(df_raw, feature_list)

    df_feat_with_placeholder = df_feat.copy()
    df_feat_with_placeholder["bad"] = 0  # placeholder — woebin_ply needs target col present
    df_woe = sc.woebin_ply(df_feat_with_placeholder, bins)

    locked_woe_cols = [f + "_woe" for f in feature_list]
    missing_woe = [c for c in locked_woe_cols if c not in df_woe.columns]
    if missing_woe:
        raise ValueError(f"WOE transform produced no columns for: {missing_woe}")

    X_score = df_woe[locked_woe_cols]

    score_df = sc.scorecard_ply(df_feat, card, print_step=0)

    score_df["bins"] = (
        10 - kbd.transform(score_df[["score"]]).astype(int).flatten()
    ).clip(1, 10)  # guard against edge values outside training range

    score_df["probability"] = model.predict_proba(X_score)[:, 1].round(6)

    score_df = pd.concat(
        [df_meta.reset_index(drop=True), score_df.reset_index(drop=True)], axis=1
    )

    if not score_df.empty:
        print(
            "Scored %d rows | score range: [%d, %d] | mean prob: %.4f",
            len(score_df), score_df["score"].min(), score_df["score"].max(),
            score_df["probability"].mean(),
        )
    return score_df

artifacts = {
    "model": pickle.load(open(f"{artifacts_dir}/{base_seg}_model.pkl", "rb")),
    "card": pickle.load(open(f"{artifacts_dir}/{base_seg}_card.pkl", "rb")),
    "bins": pickle.load(open(f"{artifacts_dir}/{base_seg}_bins.pkl", "rb")),
    "kbd": pickle.load(open(f"{artifacts_dir}/{base_seg}_kbd.pkl", "rb")),  # Load fitted kbd
    "features": (
        pd.read_csv(f"{artifacts_dir}/{base_seg}_final_features.csv")["0"]
        .str.replace("_woe", "")
        .tolist()
    ),
}


pool_scores = score_pool(pool.copy(), artifacts, out_dir)

In [ ]:
pool_scores.rename(columns={'p_date': 'base_date'}, inplace=True)

pool_scores.describe()

In [ ]:
for c in df.columns:
    print(c)
    if c == 'base_date':
        df[c] = pd.to_datetime(df[c])

    elif c == 'userid':
        df[c] = df[c].astype('str')

    elif c == 'borrowerid':
        df[c] = df[c].astype('str')

    elif c == 'loanid':
        df[c] = df[c].astype('str')

    else:
        df[c] = df[c].astype('float64')

print(df.dtypes)


In [ ]:
for c in pool_scores.columns:
    print(c)
    if c == 'base_date':
        pool_scores[c] = pd.to_datetime(pool_scores[c])

    elif c == 'userid':
        pool_scores[c] = pool_scores[c].astype('str')

    elif c == 'borrowerid':
        pool_scores[c] = pool_scores[c].astype('str')

    elif c == 'loanid':
        pool_scores[c] = pool_scores[c].astype('str')

    else:
        pool_scores[c] = pool_scores[c].astype('float64')
print(pool_scores.dtypes)


In [ ]:
import pandas as pd

df_combined = pd.concat([df, pool_scores], ignore_index=True)

In [ ]:
df_combined = df_combined.sort_values(by=['base_date', 'loanid'])
df_combined.drop_duplicates(subset=['base_date', 'loanid'], inplace=True)

In [ ]:
for c in df_combined.columns:
    print(c, df_combined[c].map(type).unique())

In [ ]:
out_path = OUT_DIR / "lease_collection_daily.parquet"
df_combined.to_parquet(out_path, index=False)
print("Saved", len(df_combined), "rows to", out_path)